In [2]:
import os
import json
from google import genai
from google.genai import types
from google.api_core import retry
from pydantic import BaseModel, Field


client = genai.Client(api_key='')
MODEL_ID = "gemini-2.5-flash"

def call_gemini(prompt, config=None):

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=config or types.GenerateContentConfig(
                temperature=0.7,
                top_p=0.95,
                top_k=40
            )
        )
        return response.text
    except Exception as e:
        return f"Error occurred: {str(e)}"



class LocationEntity(BaseModel):
    """Schema for extracting travel details."""
    city: str = Field(description="The name of the city")
    country: str = Field(description="The name of the country")
    context: str = Field(description="A brief summary of why the user is going there")

def get_structured_location(text: str):
    print("\n--- Running Structured Output Extraction ---")
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=f"Extract location info from: {text}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=LocationEntity, # Force output to match our Pydantic model
            temperature=0.0 # Deterministic
        )
    )
    # Gemini's SDK parses the JSON back into the Pydantic object automatically
    return response.parsed



def get_weather(location: str) -> str:
    """Mock weather API."""
    print(f"[*] Tool executing: get_weather for {location}")
    return "Sunny, 24°C"

def get_exchange_rate(currency: str) -> str:
    """Mock currency exchange API."""
    print(f"[*] Tool executing: get_exchange_rate for {currency}")
    return "1 USD = 2.65 GEL"

# List of tools for the model
tools = [get_weather, get_exchange_rate]

def run_agent_task(prompt: str):
    print(f"\n--- Running Tool Calling Loop ---")
    print(f"User Request: {prompt}")
    
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=tools,
            temperature=0.2
        )
    )
    
    print("\nFinal Result:")
    print(response.text)



if __name__ == "__main__":
    # Test Part 2: Structured Output
    text = "I'm heading to Tbilisi, Georgia for a 2-week hiking trip."
    data = get_structured_location(text)
    print(f"Extracted: {data.city}, {data.country} | Context: {data.context}")

    # Test Part 3: Tool Calling
    # This prompt requires calling BOTH tools to answer completely
    run_agent_task("I'm going to Tbilisi. What's the weather and what is the exchange rate for GEL?")


--- Running Structured Output Extraction ---
Extracted: Tbilisi, Georgia | Context: a 2-week hiking trip

--- Running Tool Calling Loop ---
User Request: I'm going to Tbilisi. What's the weather and what is the exchange rate for GEL?
[*] Tool executing: get_weather for Tbilisi
[*] Tool executing: get_exchange_rate for GEL

Final Result:
The weather in Tbilisi is sunny and 24°C. The exchange rate is 1 USD = 2.65 GEL.
